# 02. 저장된 raw 데이터 전처리
인터넷/API 호출 없이 반복 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'preprocess.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.preprocess import build_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
final_df_raw = pd.read_parquet(RAW_DIR / 'final_df_raw.parquet')
collection_missing_df = pd.read_csv(RAW_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')
print(final_df_raw.shape, collection_missing_df.shape, sp500_universe.shape)

(1663038, 8) (21, 7) (729, 4)


In [3]:
# --- 알려진 데이터 오염 종목 제외 (2026-08-29) ---
# PARA: raw 가격 시계열의 상당 구간이 주식이 아니라 채권 가격으로 추정되는 값으로 오염됨
#   (액면가 100 근처 저변동성, 거래량 한 자릿수대, 가격이 정수×1000 형태로 저장
#    — 회사채 가격 시스템의 전형적 특징). 발행사 식별자 충돌 등으로 채권 시계열이
#    잘못 병합된 것으로 추정. 오염 경계가 불명확해 부분 복구 대신 종목 전체 제외.
#    다운스트림 영향(제외 전 측정): rebalance_60df 안정그룹 스냅샷 기준 18/12,806행(0.14%).
EXCLUDED_TICKERS = ['PARA']

n_before = len(final_df_raw)
final_df_raw = final_df_raw[~final_df_raw['Ticker'].isin(EXCLUDED_TICKERS)].reset_index(drop=True)
print(f'제외 적용: {n_before - len(final_df_raw):,}행 제거 ({EXCLUDED_TICKERS})')

# 감사 추적: final_missing_df에도 기록 (수집 실패 21종목과는 fail_stage로 구분)
excluded_record = pd.DataFrame({
    'ticker': EXCLUDED_TICKERS,
    'company': sp500_universe.set_index('ticker')['company'].reindex(EXCLUDED_TICKERS).fillna('Unknown').values,
    'sector': sp500_universe.set_index('ticker')['sector'].reindex(EXCLUDED_TICKERS).fillna('Unknown').values,
    'fail_stage': 'data_quality',
    'fail_reason': '채권 가격 오염 추정 — 부분 구간 액면가 근접 저변동성 패턴, 상세는 위 주석',
})
collection_missing_df = pd.concat([collection_missing_df, excluded_record], ignore_index=True)

제외 적용: 1,350행 제거 (['PARA'])


In [4]:
final_df, coverage_df, final_missing_df = build_ml_dataset(
    final_df_raw, universe=sp500_universe, final_missing_df=collection_missing_df
)
display(final_df.head())
display(coverage_df.head())
display(final_missing_df.head())

,Date,Ticker,Open,High,Low,Close,Volume,source
0,2016-01-04,A,37.751924,37.871448,37.089932,37.411732,3287300,yahoo
1,2016-01-05,A,37.448518,37.650795,37.089940,37.283020,2587200,yahoo
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo
4,2016-01-08,A,36.060163,36.510683,35.370588,35.480919,3736700,yahoo


,Ticker,n_rows,actual_start_date,actual_end_date,n_price_imputed,n_ohlc_inconsistent,n_volume_missing,sources,expected_rows_10y,coverage_10y,short_history,has_quality_issue,company,sector
0,A,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Agilent Technologies,Health Care
1,AABA,948,2016-01-04,2019-11-06,0,0,0,tiingo,2738,0.3462,True,False,AABA,Unknown
2,AAL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,AAL,Unknown
3,AAP,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,AAP,Unknown
4,AAPL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Apple Inc.,Information Technology


,Ticker,company,sector,fail_stage,fail_reason,n_rows_raw,n_rows_final
0,ADS,ADS,Unknown,collection,yahoo: empty response | chart: HTTP 404 | tiin...,0,0
1,BBBY,BBBY,Unknown,collection,yahoo: empty response | chart: HTTP 400 | tiin...,0,0
2,BK,BK,Unknown,collection,yahoo: empty response | chart: HTTP 404,0,0
3,CBS,CBS,Unknown,collection,yahoo: empty response | chart: HTTP 404 | tiin...,0,0
4,CCE,CCE,Unknown,collection,yahoo: empty response | chart: no data | tiing...,0,0


In [5]:
final_df.to_parquet(PROCESSED_DIR / 'final_df.parquet', index=False)
coverage_df.to_csv(PROCESSED_DIR / 'coverage_df.csv', index=False)
final_missing_df.to_csv(PROCESSED_DIR / 'final_missing_df.csv', index=False)
print(f'processed 저장 완료: {PROCESSED_DIR}')

processed 저장 완료: /Users/choedasom/lab_middle_project/data/processed
